# Pipe CFD Post-Processing Workflow

This notebook builds the reusable CFD analysis workflow that will later be applied to `segment_test_V2`, patient-specific arterial segments, and Circle of Willis CFD simulations.

The straight pipe has already been validated. Here it is used as a controlled reference case for the analysis pattern: discover OpenFOAM outputs, load sampled fields and patch averages, compute pressure drop, flow rate, and hydraulic resistance, compare with Poiseuille theory, and save publication-ready figures and CSV summaries.

In [ ]:
from pathlib import Path
import csv
import math
import re

try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "scripts" else Path.cwd().resolve()
CASE_DIR = ROOT / "openfoam" / "pipe"
OUT_DIR = ROOT / "output" / "pipe" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RHO = 1060.0      # kg/m3; OpenFOAM incompressible p is kinematic pressure and is converted to Pa
MU = 3.5e-3       # Pa s
RADIUS = 0.002    # m
LENGTH = 0.040    # m
AREA_ANALYTICAL = math.pi * RADIUS**2

print(f"Case directory: {CASE_DIR}")
print(f"Output directory: {OUT_DIR}")

In [ ]:
def latest_time_dir(parent):
    if not parent.exists():
        return None
    dirs = [p for p in parent.iterdir() if p.is_dir()]
    if not dirs:
        return None
    def key(path):
        try:
            return float(path.name)
        except ValueError:
            return -1.0
    return max(dirs, key=key)


def write_csv(path, rows, fieldnames=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        for row in rows:
            for key in row:
                if key not in fieldnames:
                    fieldnames.append(key)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def show_rows(rows, n=5):
    if pd is not None:
        return pd.DataFrame(rows).head(n)
    for row in rows[:n]:
        print(row)
    return None


def read_xy(path):
    if not path.exists():
        return [], []
    header = None
    data = []
    for line in path.read_text().splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith("#"):
            tokens = stripped.strip("#").split()
            if tokens and tokens[0] == "distance":
                header = tokens
            continue
        data.append([float(value) for value in stripped.split()])
    if header is None and data:
        header = [f"col_{i}" for i in range(len(data[0]))]
    return header or [], [dict(zip(header, row)) for row in data]


def read_patch_average(path):
    if not path.exists():
        return None
    area = None
    field = None
    value = None
    time = None
    for line in path.read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("# Area"):
            area = float(stripped.split(":", 1)[1])
        elif stripped.startswith("# Time"):
            parts = stripped.split()
            field = parts[-1] if len(parts) >= 3 else None
        elif stripped and not stripped.startswith("#"):
            parts = stripped.split(None, 1)
            time = float(parts[0])
            raw = parts[1].strip()
            if raw.startswith("("):
                value = [float(x) for x in raw.strip("()").split()]
            else:
                value = float(raw)
    return {"path": str(path), "area": area, "field": field, "time": time, "value": value}


def read_wss(path):
    if not path.exists():
        return None
    for line in path.read_text().splitlines():
        stripped = line.strip()
        if stripped and not stripped.startswith("#"):
            parts = stripped.split("\t")
            if len(parts) >= 4:
                def vec(text):
                    return [float(x) for x in text.strip().strip("()").split()]
                component_min = vec(parts[2])
                component_max = vec(parts[3])
                return {
                    "time": float(parts[0]),
                    "patch": parts[1],
                    "component_min_vector": component_min,
                    "component_max_vector": component_max,
                    "component_min_vector_magnitude": math.sqrt(sum(v*v for v in component_min)),
                    "component_max_vector_magnitude": math.sqrt(sum(v*v for v in component_max)),
                }
    return None


def linreg(x, y):
    xm = sum(x) / len(x)
    ym = sum(y) / len(y)
    ssx = sum((value - xm) ** 2 for value in x)
    slope = sum((x[i] - xm) * (y[i] - ym) for i in range(len(x))) / ssx
    intercept = ym - slope * xm
    pred = [slope * value + intercept for value in x]
    sst = sum((value - ym) ** 2 for value in y)
    sse = sum((y[i] - pred[i]) ** 2 for i in range(len(y)))
    r2 = 1 - sse / sst if sst else float("nan")
    return slope, intercept, r2, pred


def trapz(x, y):
    return sum(0.5 * (y[i] + y[i-1]) * (x[i] - x[i-1]) for i in range(1, len(x)))

In [ ]:
def svg_plot(path, series, xlabel, ylabel, title, width=880, height=560):
    margin = {"l": 86, "r": 28, "t": 58, "b": 76}
    xs = [p[0] for s in series for p in s["points"] if math.isfinite(p[0]) and math.isfinite(p[1])]
    ys = [p[1] for s in series for p in s["points"] if math.isfinite(p[0]) and math.isfinite(p[1])]
    if not xs or not ys:
        return
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    if xmin == xmax:
        xmin, xmax = xmin - 1, xmax + 1
    if ymin == ymax:
        ymin, ymax = ymin - 1, ymax + 1
    xmin -= 0.04 * (xmax - xmin)
    xmax += 0.04 * (xmax - xmin)
    ymin -= 0.08 * (ymax - ymin)
    ymax += 0.08 * (ymax - ymin)
    pw = width - margin["l"] - margin["r"]
    ph = height - margin["t"] - margin["b"]
    def sx(x): return margin["l"] + (x - xmin) / (xmax - xmin) * pw
    def sy(y): return margin["t"] + (ymax - y) / (ymax - ymin) * ph
    colors = ["#0B6E69", "#B23A48", "#255C99", "#D49A2A"]
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', f'<text x="{width/2}" y="30" text-anchor="middle" font-family="Arial" font-size="22" font-weight="700">{title}</text>']
    for i in range(6):
        tx = xmin + i * (xmax - xmin) / 5
        px = sx(tx)
        out.append(f'<line x1="{px:.2f}" y1="{margin["t"]}" x2="{px:.2f}" y2="{height-margin["b"]}" stroke="#e7e7e7"/>')
        out.append(f'<text x="{px:.2f}" y="{height-margin["b"]+24}" text-anchor="middle" font-family="Arial" font-size="12">{tx:.3g}</text>')
    for i in range(6):
        ty = ymin + i * (ymax - ymin) / 5
        py = sy(ty)
        out.append(f'<line x1="{margin["l"]}" y1="{py:.2f}" x2="{width-margin["r"]}" y2="{py:.2f}" stroke="#e7e7e7"/>')
        out.append(f'<text x="{margin["l"]-12}" y="{py+4:.2f}" text-anchor="end" font-family="Arial" font-size="12">{ty:.3g}</text>')
    out.append(f'<line x1="{margin["l"]}" y1="{height-margin["b"]}" x2="{width-margin["r"]}" y2="{height-margin["b"]}" stroke="#222" stroke-width="1.5"/>')
    out.append(f'<line x1="{margin["l"]}" y1="{margin["t"]}" x2="{margin["l"]}" y2="{height-margin["b"]}" stroke="#222" stroke-width="1.5"/>')
    out.append(f'<text x="{width/2}" y="{height-24}" text-anchor="middle" font-family="Arial" font-size="15">{xlabel}</text>')
    out.append(f'<text x="22" y="{height/2}" transform="rotate(-90 22 {height/2})" text-anchor="middle" font-family="Arial" font-size="15">{ylabel}</text>')
    for idx, s in enumerate(series):
        color = s.get("color", colors[idx % len(colors)])
        dash = ' stroke-dasharray="7 5"' if s.get("dash") else ""
        pts = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y in s["points"] if math.isfinite(x) and math.isfinite(y))
        out.append(f'<polyline fill="none" stroke="{color}" stroke-width="3"{dash} points="{pts}"/>')
        if s.get("markers"):
            for x, y in s["points"]:
                out.append(f'<circle cx="{sx(x):.2f}" cy="{sy(y):.2f}" r="3" fill="{color}"/>')
    lx = width - margin["r"] - 250
    ly = margin["t"] + 8
    for idx, s in enumerate(series):
        color = s.get("color", colors[idx % len(colors)])
        y = ly + idx * 24
        out.append(f'<line x1="{lx}" y1="{y}" x2="{lx+34}" y2="{y}" stroke="{color}" stroke-width="3"/>')
        out.append(f'<text x="{lx+44}" y="{y+5}" font-family="Arial" font-size="13">{s["label"]}</text>')
    out.append("</svg>")
    path.write_text("\n".join(out))


def svg_bar(path, labels, values, ylabel, title, width=820, height=540):
    margin = {"l": 90, "r": 36, "t": 58, "b": 100}
    ymax = max(values) * 1.18 if values else 1
    pw = width - margin["l"] - margin["r"]
    ph = height - margin["t"] - margin["b"]
    barw = pw / max(1, len(values)) * 0.56
    colors = ["#0B6E69", "#B23A48", "#255C99", "#D49A2A"]
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', f'<text x="{width/2}" y="30" text-anchor="middle" font-family="Arial" font-size="22" font-weight="700">{title}</text>']
    for i in range(6):
        val = i * ymax / 5
        y = margin["t"] + (ymax - val) / ymax * ph
        out.append(f'<line x1="{margin["l"]}" y1="{y:.2f}" x2="{width-margin["r"]}" y2="{y:.2f}" stroke="#e7e7e7"/>')
        out.append(f'<text x="{margin["l"]-12}" y="{y+4:.2f}" text-anchor="end" font-family="Arial" font-size="12">{val:.3g}</text>')
    out.append(f'<line x1="{margin["l"]}" y1="{height-margin["b"]}" x2="{width-margin["r"]}" y2="{height-margin["b"]}" stroke="#222"/>')
    out.append(f'<line x1="{margin["l"]}" y1="{margin["t"]}" x2="{margin["l"]}" y2="{height-margin["b"]}" stroke="#222"/>')
    for i, (label, value) in enumerate(zip(labels, values)):
        cx = margin["l"] + (i + 0.5) * pw / len(values)
        h = value / ymax * ph
        y = height - margin["b"] - h
        out.append(f'<rect x="{cx-barw/2:.2f}" y="{y:.2f}" width="{barw:.2f}" height="{h:.2f}" fill="{colors[i%len(colors)]}"/>')
        out.append(f'<text x="{cx:.2f}" y="{y-8:.2f}" text-anchor="middle" font-family="Arial" font-size="12">{value:.3g}</text>')
        out.append(f'<text x="{cx:.2f}" y="{height-margin["b"]+24}" text-anchor="middle" font-family="Arial" font-size="13">{label}</text>')
    out.append(f'<text x="24" y="{height/2}" transform="rotate(-90 24 {height/2})" text-anchor="middle" font-family="Arial" font-size="15">{ylabel}</text>')
    out.append("</svg>")
    path.write_text("\n".join(out))

## Step 1: Discover Available CFD Outputs

The notebook searches the pipe case for `postProcessing/`, `sampleDict` files, patch averages, wall shear stress outputs, CSV exports, and ParaView state/export files. Missing files are reported but do not stop the notebook.

In [ ]:
sample_time = latest_time_dir(CASE_DIR / "postProcessing" / "sampleDict")
wss_time = latest_time_dir(CASE_DIR / "postProcessing" / "wallShearStress")
paths = {
    "postProcessing/": CASE_DIR / "postProcessing",
    "centreline.xy": sample_time / "centreline.xy" if sample_time else CASE_DIR / "postProcessing/sampleDict/<time>/centreline.xy",
    "outletRadial.xy": sample_time / "outletRadial.xy" if sample_time else CASE_DIR / "postProcessing/sampleDict/<time>/outletRadial.xy",
    "inlet patchAverage(U)": CASE_DIR / "postProcessing/patchAverage(patch=inlet,fields=(U))/144/surfaceFieldValue.dat",
    "outlet patchAverage(U)": CASE_DIR / "postProcessing/patchAverage(patch=outlet,fields=(U))/144/surfaceFieldValue.dat",
    "inlet patchAverage(p)": CASE_DIR / "postProcessing/patchAverage(patch=inlet,fields=(p))/144/surfaceFieldValue.dat",
    "outlet patchAverage(p)": CASE_DIR / "postProcessing/patchAverage(patch=outlet,fields=(p))/144/surfaceFieldValue.dat",
    "wallShearStress.dat": wss_time / "wallShearStress.dat" if wss_time else CASE_DIR / "postProcessing/wallShearStress/<time>/wallShearStress.dat",
    "ParaView state/export": CASE_DIR / "postProcessing/pipe_postprocessing.pvsm",
}
purposes = {
    "postProcessing/": "OpenFOAM post-processing directory containing sampled lines and patch reductions.",
    "centreline.xy": "Sampled centreline pressure and velocity along the vessel axis.",
    "outletRadial.xy": "Sampled outlet diameter profile for velocity-profile analysis.",
    "inlet patchAverage(U)": "Area-average inlet velocity vector for inlet flow rate.",
    "outlet patchAverage(U)": "Area-average outlet velocity vector for outlet flow rate and mass conservation.",
    "inlet patchAverage(p)": "Area-average inlet pressure for pressure drop.",
    "outlet patchAverage(p)": "Area-average outlet pressure for pressure drop.",
    "wallShearStress.dat": "Wall shear stress range on the wall patch.",
    "ParaView state/export": "Saved ParaView visualization/export state.",
}
discovery = [{"File": name, "Exists": path.exists(), "Purpose": purposes[name], "Path": str(path)} for name, path in paths.items()]
write_csv(OUT_DIR / "discovered_files.csv", discovery)
show_rows(discovery, len(discovery))

missing = [row["File"] for row in discovery if not row["Exists"]]
if missing:
    print("\nMissing files:", ", ".join(missing))
    print("Generate sampled lines with: postProcess -func sampleDict -latestTime")
    print("Generate patch averages with: postProcess -func 'patchAverage(patch=<patch>,fields=(U p))' -latestTime")
    print("Generate wall shear stress with: simpleFoam -postProcess -func wallShearStress -latestTime")
else:
    print("\nAll expected pipe post-processing files were found.")

## Step 2: Load and Inspect Data

`centreline.xy` is a line sample along the pipe axis. It represents the axial pressure and velocity field and is used to estimate pressure linearity and gradient.

`outletRadial.xy` is a diameter sample across the outlet. It represents the radial velocity distribution and is used to compare the CFD profile with Poiseuille flow.

Patch-average files represent inlet/outlet surface averages. These are the preferred sources for network quantities because future 0D/1D models exchange pressure and flow at segment boundaries.

Units: positions are in meters, velocity is m/s, OpenFOAM incompressible `p` is kinematic pressure in m2/s2 and is converted to Pa using `rho = 1060 kg/m3`.

In [ ]:
centre_cols, centreline = read_xy(paths["centreline.xy"])
radial_cols, radial = read_xy(paths["outletRadial.xy"])
patches = {name: read_patch_average(path) for name, path in paths.items() if "patchAverage" in name}
wss = read_wss(paths["wallShearStress.dat"])

if centreline:
    write_csv(OUT_DIR / "centreline_loaded.csv", centreline, centre_cols)
    print(f"centreline.xy: {len(centreline)} samples")
    print("columns:", centre_cols)
    show_rows(centreline)
else:
    print("centreline.xy not found.")

if radial:
    write_csv(OUT_DIR / "outlet_radial_loaded.csv", radial, radial_cols)
    print(f"\noutletRadial.xy: {len(radial)} samples")
    print("columns:", radial_cols)
    show_rows(radial)
else:
    print("outletRadial.xy not found.")

print("\nPatch averages:")
for name, value in patches.items():
    print(name, value)

## Step 3: Pressure Analysis

Plot 1 shows pressure against axial position. The linear fit gives the pressure gradient `dP/dx`, while patch averages give the boundary pressure drop `DeltaP = P_inlet - P_outlet` used later for hydraulic resistance.

Pressure drop is one of the most important quantities for 0D/1D comparison because reduced-order models are pressure-flow models at their core.

In [ ]:
pressure_summary = []
delta_p_cfd = None
pressure_gradient = None

if centreline:
    z = [row["z"] for row in centreline]
    p_pa = [row["p"] * RHO for row in centreline]
    slope, intercept, r2, fit = linreg(z, p_pa)
    pressure_gradient = slope
    centreline_delta_p = p_pa[0] - p_pa[-1]
    pressure_summary.extend([
        {"metric": "centreline_pressure_gradient_dPdx", "value": slope, "units": "Pa/m"},
        {"metric": "centreline_linear_fit_R2", "value": r2, "units": "-"},
        {"metric": "centreline_deltaP", "value": centreline_delta_p, "units": "Pa"},
    ])
    svg_plot(OUT_DIR / "plot1_pressure_vs_axial_position.svg", [
        {"label": "CFD centreline pressure", "points": list(zip(z, p_pa))},
        {"label": "linear fit", "points": list(zip(z, fit)), "dash": True},
    ], "vessel length z (m)", "pressure (Pa)", "Pressure vs axial position")
    print(f"Centreline pressure is approximately linear: R2 = {r2:.6f}")
    print(f"dP/dx = {slope:.3f} Pa/m")
    print(f"centreline DeltaP = {centreline_delta_p:.6g} Pa")

pin = patches.get("inlet patchAverage(p)")
pout = patches.get("outlet patchAverage(p)")
if pin and pout and isinstance(pin["value"], float) and isinstance(pout["value"], float):
    delta_p_cfd = (pin["value"] - pout["value"]) * RHO
    pressure_summary.append({"metric": "patch_deltaP", "value": delta_p_cfd, "units": "Pa"})
    print(f"patch DeltaP = {delta_p_cfd:.6g} Pa")

write_csv(OUT_DIR / "pressure_summary.csv", pressure_summary)
show_rows(pressure_summary, len(pressure_summary))

## Step 4: Velocity Analysis

Plot 2 compares the CFD outlet profile to the analytical Poiseuille profile. The workflow computes `Umax`, area-weighted `Umean`, RMSE, and maximum deviation. For later arterial cases this same section becomes a diagnostic for non-parabolic profiles caused by curvature, bifurcations, or secondary flow.

In [ ]:
velocity_summary = []
profile_stats = {}

if radial:
    profile = []
    for row in radial:
        u_mag = math.sqrt(row["U_x"]**2 + row["U_y"]**2 + row["U_z"]**2)
        profile.append({"radius_m": row["x"], "abs_radius_m": abs(row["x"]), "U_axial_m_per_s": row["U_z"], "U_mag_m_per_s": u_mag})
    write_csv(OUT_DIR / "outlet_radial_velocity_profile.csv", profile)

    bins = {}
    for row in profile:
        bins.setdefault(round(row["abs_radius_m"], 10), []).append(row["U_axial_m_per_s"])
    rvals = sorted(bins)
    uvals = [sum(bins[r]) / len(bins[r]) for r in rvals]
    if rvals and rvals[0] > 0:
        rvals = [0.0] + rvals
        uvals = [max(uvals)] + uvals
    integral = trapz(rvals, [uvals[i] * rvals[i] for i in range(len(rvals))])
    umean_profile = 2 * integral / RADIUS**2
    umax_profile = max(row["U_axial_m_per_s"] for row in profile)

    comparison = []
    squared_errors = []
    deviations = []
    for row in profile:
        poiseuille = 2 * umean_profile * max(0.0, 1 - (row["abs_radius_m"] / RADIUS) ** 2)
        error = row["U_axial_m_per_s"] - poiseuille
        squared_errors.append(error * error)
        deviations.append(abs(error))
        comparison.append({**row, "U_poiseuille_m_per_s": poiseuille, "error_m_per_s": error})
    rmse = math.sqrt(sum(squared_errors) / len(squared_errors))
    max_deviation = max(deviations)
    profile_error_pct = 100 * rmse / umax_profile if umax_profile else float("nan")
    profile_stats = {
        "Umean_profile_m_per_s": umean_profile,
        "Umax_profile_m_per_s": umax_profile,
        "profile_RMSE_m_per_s": rmse,
        "profile_max_deviation_m_per_s": max_deviation,
        "profile_RMSE_percent_of_Umax": profile_error_pct,
    }
    velocity_summary = [{"metric": key, "value": value, "units": "%" if "percent" in key else "m/s"} for key, value in profile_stats.items()]
    write_csv(OUT_DIR / "velocity_profile_comparison.csv", comparison)
    write_csv(OUT_DIR / "velocity_summary.csv", velocity_summary)
    svg_plot(OUT_DIR / "plot2_velocity_profile_vs_radius.svg", [
        {"label": "CFD outlet profile", "points": [(row["radius_m"], row["U_axial_m_per_s"]) for row in comparison], "markers": True},
        {"label": "Poiseuille profile", "points": [(row["radius_m"], row["U_poiseuille_m_per_s"]) for row in comparison], "dash": True},
    ], "radius x (m)", "axial velocity (m/s)", "Outlet velocity profile")
    print(f"Profile is parabolic to within RMSE {rmse:.3e} m/s ({profile_error_pct:.3f}% of Umax).")
    show_rows(velocity_summary, len(velocity_summary))
else:
    print("No outlet radial profile available; skipping velocity-profile comparison.")

## Step 5: Flow Rate Analysis

Flow rate is computed as `Q = Umean * Area` using patch-average axial velocity and patch area. This is the primary quantity for 1D modelling because branch flow rates are conserved and coupled across vascular networks.

In [ ]:
flow_rows = []
for location, key in [("inlet", "inlet patchAverage(U)"), ("outlet", "outlet patchAverage(U)")]:
    data = patches.get(key)
    if data and isinstance(data["value"], list):
        area = data["area"] or AREA_ANALYTICAL
        umean = data["value"][2]
        flow_rows.append({"Location": location, "Area_m2": area, "Umean_m_per_s": umean, "Q_m3_per_s": umean * area})

write_csv(OUT_DIR / "flow_rate_summary.csv", flow_rows)
if flow_rows:
    svg_bar(OUT_DIR / "plot3_inlet_vs_outlet_flow_rate.svg", [row["Location"] for row in flow_rows], [abs(row["Q_m3_per_s"]) for row in flow_rows], "flow rate |Q| (m3/s)", "Inlet vs outlet flow rate")
    show_rows(flow_rows, len(flow_rows))
    if len(flow_rows) >= 2:
        qin = abs(flow_rows[0]["Q_m3_per_s"])
        qout = abs(flow_rows[-1]["Q_m3_per_s"])
        diff = 100 * abs(qin - qout) / ((qin + qout) / 2)
        print(f"Mass-conservation percentage difference = {diff:.6g}%")
else:
    print("Patch-average velocity data missing; cannot compute Q.")

## Step 6: Hydraulic Resistance Analysis

This is the most important section. The CFD resistance is `R_CFD = DeltaP / Q`. It is compared with analytical Poiseuille resistance for this pipe. In real arterial segments the same CFD extraction stays unchanged, while the analytical term becomes a geometry-derived reduced-order estimate.

Resistance is the key scalar that links CFD, 0D, and 1D models: CFD resolves it from fields, 0D stores it as a circuit element, and 1D predicts it from vessel geometry, fluid properties, and boundary conditions.

In [ ]:
q_cfd = None
if flow_rows:
    outlet = next((row for row in flow_rows if row["Location"] == "outlet"), flow_rows[-1])
    q_cfd = abs(outlet["Q_m3_per_s"])

u_mean_ref = abs(next((row["Umean_m_per_s"] for row in flow_rows if row["Location"] == "outlet"), profile_stats.get("Umean_profile_m_per_s", 0.1)))
q_analytical = u_mean_ref * AREA_ANALYTICAL
delta_p_analytical = 8 * MU * LENGTH * u_mean_ref / RADIUS**2
r_analytical = delta_p_analytical / q_analytical if q_analytical else float("nan")
umax_analytical = 2 * u_mean_ref
r_cfd = delta_p_cfd / q_cfd if delta_p_cfd is not None and q_cfd else None

comparison_values = {
    "DeltaP_Pa": (delta_p_cfd, delta_p_analytical),
    "Q_m3_per_s": (q_cfd, q_analytical),
    "R_Pa_s_per_m3": (r_cfd, r_analytical),
    "Umax_m_per_s": (profile_stats.get("Umax_profile_m_per_s"), umax_analytical),
}
resistance_rows = []
for quantity, (cfd, analytical) in comparison_values.items():
    error = 100 * (cfd - analytical) / analytical if cfd is not None and analytical else None
    resistance_rows.append({"Quantity": quantity, "CFD": cfd, "Analytical": analytical, "Error_percent": error})
write_csv(OUT_DIR / "resistance_comparison.csv", resistance_rows)
if r_cfd is not None:
    svg_bar(OUT_DIR / "plot4_resistance_comparison.svg", ["CFD", "Poiseuille"], [r_cfd, r_analytical], "resistance (Pa s / m3)", "Hydraulic resistance comparison")
show_rows(resistance_rows, len(resistance_rows))

## Step 7: Summary of Quantities Relevant for Future CoW Simulations

This table marks how each quantity appears in CFD, 1D, and 0D models, and whether it will be used later for Circle of Willis analysis.

In [ ]:
cow_rows = [
    {"Quantity": "Pressure", "CFD": "field or patch average", "1D": "nodal/segment pressure", "0D": "node pressure", "Used Later?": "Yes", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "absolute/relative pressure checks"},
    {"Quantity": "Pressure drop", "CFD": "inlet-outlet patch pressure", "1D": "segment pressure loss", "0D": "R*Q relation", "Used Later?": "Yes", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "primary segment comparison"},
    {"Quantity": "Flow rate", "CFD": "patch Umean*Area", "1D": "primary state variable", "0D": "branch flow", "Used Later?": "Yes", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "primary network comparison"},
    {"Quantity": "Resistance", "CFD": "DeltaP/Q", "1D": "geometry/friction closure", "0D": "network element", "Used Later?": "Yes", "Geometry Alone?": "Estimate only", "Needs BCs?": "CFD yes", "Validation Use": "main CFD to 0D/1D bridge"},
    {"Quantity": "Mean velocity", "CFD": "patch area average", "1D": "Q/A", "0D": "derived from Q/A", "Used Later?": "Yes", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "secondary flow diagnostic"},
    {"Quantity": "Peak velocity", "CFD": "sample/profile maximum", "1D": "profile assumption", "0D": "usually not explicit", "Used Later?": "Sometimes", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "profile-shape diagnostic"},
    {"Quantity": "Wall shear stress", "CFD": "wallShearStress field", "1D": "model-dependent estimate", "0D": "not resolved", "Used Later?": "Yes for CFD diagnostics", "Geometry Alone?": "No", "Needs BCs?": "Yes", "Validation Use": "CFD-only or approximate 1D comparison"},
]
write_csv(OUT_DIR / "future_cow_quantity_map.csv", cow_rows)
show_rows(cow_rows, len(cow_rows))

## Step 8: Transition from Pipe to Real Arterial Segment

The parts that remain unchanged for curved vessels, bifurcations, and Circle of Willis cases are output discovery, patch-average pressure and velocity loading, pressure-drop calculation, flow-rate calculation, resistance calculation, summary-table generation, and figure export.

The geometry-specific parts will change: curved vessels need centreline-aware axial coordinates, bifurcations need branch-wise inlet/outlet patch maps, and Circle of Willis cases need per-branch resistance summaries. The reduced-order comparison will move from a single ideal pipe formula to centreline-integrated 1D or network-level 0D estimates.

Future workflow:

CFD Segment
↓
Extract DeltaP and Q
↓
Compute R_CFD
↓
Extract centreline geometry
↓
Compute R_Poiseuille
↓
Compare
↓
Validate reduced-order model

This is the main reason the notebook exists: the pipe is a clean rehearsal for the CFD to 0D/1D comparison workflow.

In [ ]:
summary_rows = []
def add(quantity, value, units):
    summary_rows.append({"Quantity": quantity, "Value": value, "Units": units})

add("pipe_radius", RADIUS, "m")
add("pipe_length", LENGTH, "m")
add("rho_used_for_pressure_conversion", RHO, "kg/m3")
add("mu_used_for_poiseuille", MU, "Pa s")
add("pressure_gradient_centreline", pressure_gradient, "Pa/m")
add("deltaP_CFD_patch", delta_p_cfd, "Pa")
add("Q_CFD_outlet", q_cfd, "m3/s")
add("R_CFD", r_cfd, "Pa s/m3")
add("R_Poiseuille", r_analytical, "Pa s/m3")
if flow_rows and len(flow_rows) >= 2:
    qin = abs(flow_rows[0]["Q_m3_per_s"])
    qout = abs(flow_rows[-1]["Q_m3_per_s"])
    add("mass_conservation_percent_difference", 100 * abs(qin - qout) / ((qin + qout) / 2), "%")
if wss:
    add("wall_shear_stress_component_min_vector_magnitude", wss["component_min_vector_magnitude"], "kinematic units")
    add("wall_shear_stress_component_max_vector_magnitude", wss["component_max_vector_magnitude"], "kinematic units")

write_csv(OUT_DIR / "analysis_scalar_summary.csv", summary_rows)
show_rows(summary_rows, len(summary_rows))
print("\nSaved outputs:")
for path in sorted(OUT_DIR.iterdir()):
    print(path.name)